# Exp 5 — Signal Alignment: NLI + UMLS Density + UMLS Specificity

End-to-end pipeline that generates biomedical reasoning chains, extracts UMLS concepts,
scores logical entailment, and detects semantic leakage through three independent signals.

| Stage | What it does |
|-------|--------------|
| 0 | Environment setup, API keys, module imports |
| 1 | CoT Generator — 5 LLM chains + 5 prewritten test cases |
| 2 | Concept Extractor — UMLS concept linking for all 10 chains |
| 3 | NLI Entailment + Guard Signals — all chains, all step-pairs |
| 5 | UMLS Signal Computation — density + specificity for all chains |
| 6 | Signal Alignment & Visualisation |

Run cells top-to-bottom in order.
All stages (2–5) operate in full-batch mode across all 10 chains.
Stage 6 visualisations (6a–6d) use chain 1 as a drill-down example;
6e shows the comparison across all 10 chains.


## Stage 0 — Environment Setup

Clones the repo, installs dependencies, configures API keys. Run this first.


In [ ]:
# ── 0a. Clone / pull the repo and configure paths ─────────────────────────
import os, sys
from pathlib import Path

REPO_URL  = 'https://github.com/patmazza25/Biomedical-Ontology-Based-Semantic-Leakage-Detection-2'
REPO_DIR  = 'Biomedical-Ontology-Based-Semantic-Leakage-Detection-2'

if not Path(REPO_DIR).exists():
    os.system(f'git clone {REPO_URL}')
else:
    os.system(f'git -C {REPO_DIR} pull --quiet')

_cwd = Path(os.getcwd())
if (_cwd / REPO_DIR / 'utils').exists():
    PROJECT_ROOT = str(_cwd / REPO_DIR)
elif (_cwd / 'utils').exists():
    PROJECT_ROOT = str(_cwd)
elif (_cwd.parent / 'utils').exists():
    PROJECT_ROOT = str(_cwd.parent)
else:
    PROJECT_ROOT = str(_cwd / REPO_DIR)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'Python       : {sys.version.split()[0]}')
print(f'Working dir  : {os.getcwd()}')


In [ ]:
# ── 0b. Install dependencies ───────────────────────────────────────────────
!pip install openai numpy pandas scipy scikit-learn matplotlib seaborn requests ipywidgets --quiet
print('Dependencies installed.')


In [ ]:
# ── 0c. API key configuration ─────────────────────────────────────────────
import os, importlib.util
from IPython.display import display, clear_output, HTML

os.environ.setdefault('FORCE_HEURISTIC_NLI', '1')

_HAS_WIDGETS = importlib.util.find_spec('ipywidgets') is not None

if _HAS_WIDGETS:
    import ipywidgets as widgets

    # ── OpenRouter key ──────────────────────────────────────────────────────
    _or_box = widgets.Password(
        placeholder='sk-or-v1-…  (get yours free at openrouter.ai)',
        layout=widgets.Layout(width='520px'),
    )
    _or_btn = widgets.Button(
        description='Set Key', button_style='primary',
        icon='check', layout=widgets.Layout(width='110px'),
    )
    _or_out = widgets.Output()

    def _apply_or(_b):
        with _or_out:
            clear_output()
            key = _or_box.value.strip()
            if key:
                os.environ['OPENROUTER_API_KEY'] = key
                print(f'  ✓ OpenRouter key set ({len(key)} chars)')
            else:
                print('  ✗ Paste your OpenRouter key above, then click Set Key.')

    _or_btn.on_click(_apply_or)

    # ── UMLS key ───────────────────────────────────────────────────────────
    _umls_box = widgets.Password(
        placeholder='UMLS API key  (optional — enables density + specificity signals)',
        layout=widgets.Layout(width='520px'),
    )
    _umls_btn = widgets.Button(
        description='Set Key', button_style='info',
        icon='check', layout=widgets.Layout(width='110px'),
    )
    _umls_out = widgets.Output()

    def _apply_umls(_b):
        with _umls_out:
            clear_output()
            key = _umls_box.value.strip()
            if key:
                os.environ['UMLS_API_KEY'] = key
                print(f'  ✓ UMLS key set ({len(key)} chars)')
            else:
                print('  ✗ Paste your UMLS API key above, then click Set Key.')

    _umls_btn.on_click(_apply_umls)

    display(HTML('🔑 OpenRouter API Key (required for CoT generation)'))
    display(widgets.HBox([_or_box, _or_btn]))
    display(_or_out)
    display(HTML('Get a free key at openrouter.ai'))
    display(HTML('🔑 UMLS API Key (optional — enables UMLS density + specificity signals)'))
    display(widgets.HBox([_umls_box, _umls_btn]))
    display(_umls_out)
    display(HTML('Get a key at uts.nlm.nih.gov'))
else:
    os.environ.setdefault('OPENROUTER_API_KEY', '')
    os.environ.setdefault('UMLS_API_KEY', '')
    print('ipywidgets not found — set keys manually:')
    print('  os.environ["OPENROUTER_API_KEY"] = "sk-or-v1-..."')
    print('  os.environ["UMLS_API_KEY"]        = "your-umls-key"')


In [ ]:
# ── 0d. Import all pipeline modules ────────────────────────────────────────
import warnings, json, time, traceback
from collections import Counter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

warnings.filterwarnings('ignore')

_ok = {}
for mod, sym in [
    ('utils.cot_generator',           'generate'),
    ('utils.concept_extractor_v2',    'extract_concepts'),
    ('utils.hybrid_checker',          'build_entailment_records'),
    ('utils.guards',                  'derive_guards'),
    ('utils.umls_api_linker',         'is_configured'),
    ('utils.umls_density_scorer',     'score_density'),
    ('utils.umls_specificity_scorer', 'score_specificity'),
]:
    try:
        m = __import__(mod, fromlist=[sym])
        _ok[mod] = True
        print(f'  ✓ {mod}')
    except Exception as e:
        _ok[mod] = False
        print(f'  ✗ {mod}: {e}')

from utils.cot_generator           import generate as generate_cot
from utils.hybrid_checker          import build_entailment_records
from utils.guards                  import derive_guards, GuardConfig, lexical_jaccard
from utils.umls_api_linker         import is_configured as umls_configured
from utils.umls_density_scorer     import score_density
from utils.umls_specificity_scorer import score_specificity

# concept_extractor_v2 reads UMLS_API_KEY at call-time (no stale import issue).
# Fall back to the original extractor if v2 is not present.
try:
    from utils.concept_extractor_v2 import extract_concepts
    _extractor_version = 'v2 (standalone REST, call-time key)'
except ImportError:
    from utils.concept_extractor import extract_concepts
    _extractor_version = 'v1 (original)'
    # Patch module-level key so the original extractor sees the env value
    import utils.umls_api_linker as _linker
    _linker.UMLS_API_KEY = os.environ.get('UMLS_API_KEY', '') or _linker.UMLS_API_KEY

GUARD_CFG = GuardConfig()

print()
print(f'  Concept extractor : {_extractor_version}')
print(f'  OpenRouter ready  : {bool(os.environ.get("OPENROUTER_API_KEY"))}')
print(f'  Anthropic ready   : {bool(os.environ.get("ANTHROPIC_API_KEY"))}')
print(f'  UMLS configured   : {umls_configured()}')
print(f'  Heuristic NLI     : {os.environ.get("FORCE_HEURISTIC_NLI", "1") == "1"}')
print()
print('  ↑ Re-run this cell after entering your OpenRouter key above.')
if not umls_configured():
    print()
    print('  ℹ  UMLS not configured — density and specificity signals will show configured=False.')
    print('     NLI signal and visualisations will still work fully.')


## Stage 1 — CoT Generator

Calls an LLM to produce numbered reasoning steps for a biomedical question.
If no API key is set, falls back to 5 generic template steps (provider = local).


In [ ]:
# ── 1a. Configuration ───────────────────────────────────────────────────────
import os
from IPython.display import display, HTML
import importlib.util

_HAS_WIDGETS = importlib.util.find_spec('ipywidgets') is not None

_MODEL_OPTIONS = {
    'Claude Haiku (fast, recommended)':  'anthropic/claude-haiku-4-5',
    'GPT-4o Mini (OpenAI)':              'openai/gpt-4o-mini',
    'Gemini Flash (Google)':             'google/gemini-flash-1.5',
    'Llama 3.3 70B (free tier)':         'meta-llama/llama-3.3-70b-instruct:free',
}

# ── Batch toggle ─────────────────────────────────────────────────────────────
BATCH_TEST = True

if _HAS_WIDGETS:
    import ipywidgets as widgets

    _model_dropdown = widgets.Dropdown(
        options=list(_MODEL_OPTIONS.keys()),
        value='Claude Haiku (fast, recommended)',
        description='Model:',
        layout=widgets.Layout(width='420px'),
    )
    _batch_toggle = widgets.ToggleButton(
        value=True,
        description='Batch test (10 chains)',
        button_style='info',
        icon='list',
        layout=widgets.Layout(width='220px'),
    )
    display(HTML('Model selection — leave as-is if unsure'))
    display(_model_dropdown)
    display(HTML('Batch mode — 5 LLM chains + 5 prewritten test cases'))
    display(_batch_toggle)

    MODEL_ID   = _MODEL_OPTIONS[_model_dropdown.value]
    BATCH_TEST = _batch_toggle.value

    def _on_model_change(change):
        global MODEL_ID
        MODEL_ID = _MODEL_OPTIONS[change['new']]
        print(f'  Model set to: {MODEL_ID}')

    def _on_batch_change(change):
        global BATCH_TEST
        BATCH_TEST = change['new']
        print(f'  Batch mode: {BATCH_TEST}')

    _model_dropdown.observe(_on_model_change, names='value')
    _batch_toggle.observe(_on_batch_change, names='value')
else:
    MODEL_ID   = 'anthropic/claude-haiku-4-5'
    BATCH_TEST = True

PREFER = 'openrouter'
TEST_QUESTION = (
    'Does aspirin reduce the risk of myocardial infarction '
    'in patients with cardiovascular disease?'
)

print(f'Model      : {MODEL_ID}')
print(f'Batch mode : {BATCH_TEST}')
print(f'Question   : {TEST_QUESTION} (single mode only)')


In [ ]:
# ── 1b. Run CoT generation ──────────────────────────────────────────────────
# Reload cot_generator and explicitly patch its module-level API key variables
# so generate_cot picks up the key set in Stage 0c (not the stale import-time value).
import importlib
import utils.cot_generator as _cg_mod
importlib.reload(_cg_mod)

_cg_mod.OPENROUTER_API_KEY = os.environ.get('OPENROUTER_API_KEY', '')
_cg_mod.OPENROUTER_READY   = bool(_cg_mod.OPENROUTER_API_KEY)
_cg_mod.ANTHROPIC_API_KEY  = os.environ.get('ANTHROPIC_API_KEY', '')
_cg_mod.ANTHROPIC_READY    = bool(_cg_mod.ANTHROPIC_API_KEY)

from utils.cot_generator import generate as generate_cot

# ── Preflight: verify key + model before batch ────────────────────────────
def _openrouter_preflight(api_key, model_id):
    try:
        from openai import OpenAI
        client = OpenAI(api_key=api_key, base_url='https://openrouter.ai/api/v1')
        client.chat.completions.create(
            model=model_id, max_tokens=5,
            messages=[{'role': 'user', 'content': 'hi'}],
            extra_headers={
                'HTTP-Referer': 'https://github.com/biomedical-semantic-leakage',
                'X-Title': 'Biomedical Semantic Leakage Detection',
            },
        )
        return True, None
    except Exception as e:
        return False, str(e)

_key = os.environ.get('OPENROUTER_API_KEY', '')
if _key:
    print(f'Checking OpenRouter key + model ({MODEL_ID})...')
    _pf_ok, _pf_err = _openrouter_preflight(_key, MODEL_ID)
    if _pf_ok:
        print(f'  ✓ OpenRouter ready — model accessible\n')
    else:
        print(f'  ✗ OpenRouter error: {_pf_err}')
        print(f'\n  Common causes:')
        print(f'    • insufficient credits / 402  → switch to a FREE model in Stage 1a')
        print(f'    • invalid api key / 401       → re-paste your key in Stage 0c')
        print(f'    • model not found / 404       → model ID may have changed\n')
else:
    print('  ✗ No API key set — paste your OpenRouter key in Stage 0c and click Set Key.\n')

# ── CoT generation ────────────────────────────────────────────────────────
BATCH_LLM_QUESTIONS = [
    'Does aspirin reduce the risk of myocardial infarction in patients with cardiovascular disease?',
    'How do ACE inhibitors reduce blood pressure in hypertension?',
    'What is the role of TNF-alpha in rheumatoid arthritis joint destruction?',
    'How does chronic kidney disease progress to end-stage renal disease?',
    'What is the mechanism of action of selective serotonin reuptake inhibitors?',
]

LLM_CHAINS = []

if BATCH_TEST:
    print(f'Batch mode — generating {len(BATCH_LLM_QUESTIONS)} LLM chains\n')
    print('=' * 70)

    for qi, q in enumerate(BATCH_LLM_QUESTIONS):
        print(f'\n[{qi+1}/{len(BATCH_LLM_QUESTIONS)}] {q}')
        print('-' * 60)
        t0 = time.time()
        try:
            cot     = generate_cot(q, prefer=PREFER, model=MODEL_ID)
            elapsed = round(time.time() - t0, 2)
            steps   = cot['steps']

            for si, step in enumerate(steps, 1):
                print(f'  Step {si:2d}: {step}')

            print(f'\n  Provider: {cot["provider"]}  |  Model: {cot["model"]}  |  {len(steps)} steps  |  {elapsed}s')

            LLM_CHAINS.append({
                'id':       f'llm_{qi+1}',
                'label':    f'LLM {qi+1} — {q[:55]}',
                'source':   'llm',
                'question': q,
                'expect':   'unknown',
                'steps':    steps,
                'provider': cot['provider'],
                'model':    cot['model'],
            })

            if cot['provider'] == 'local':
                print('  ⚠  provider=local — LLM call failed despite preflight passing.')
                print(f'     OPENROUTER_READY in module: {_cg_mod.OPENROUTER_READY}')
                print(f'     Key in module (len): {len(_cg_mod.OPENROUTER_API_KEY or "")}')

        except Exception as e:
            elapsed = round(time.time() - t0, 2)
            print(f'  ERROR after {elapsed}s: {e}')

        time.sleep(0.3)

    print('\n' + '=' * 70)
    print(f'Generated {len(LLM_CHAINS)}/{len(BATCH_LLM_QUESTIONS)} LLM chains successfully.')

    if LLM_CHAINS:
        STEPS         = LLM_CHAINS[0]['steps']
        PROVIDER      = LLM_CHAINS[0]['provider']
        MODEL_ID_USED = LLM_CHAINS[0]['model']
        print(f'\nSTEPS set to chain 1 for single-chain stages (2–5).')
        print('All stages (2–5) process all 10 chains in full-batch mode.')
    else:
        print('\n⚠  No chains generated — check the preflight error above.')
        STEPS = []
        PROVIDER = 'local'
        MODEL_ID_USED = 'none'

else:
    t0  = time.time()
    cot = generate_cot(TEST_QUESTION, prefer=PREFER, model=MODEL_ID)
    elapsed = round(time.time() - t0, 2)

    STEPS         = cot['steps']
    PROVIDER      = cot['provider']
    MODEL_ID_USED = cot['model']

    print(f'Provider : {PROVIDER}  |  Model : {MODEL_ID_USED}  |  Time : {elapsed}s')
    print(f'Steps    : {len(STEPS)}')
    print()
    for i, step in enumerate(STEPS, 1):
        print(f'  Step {i:2d}: {step}')

    if PROVIDER == 'local':
        print()
        print("⚠  provider='local' — API call failed.")
        print(f'   OPENROUTER_READY in module: {_cg_mod.OPENROUTER_READY}')
        print(f'   Key in module (len): {len(_cg_mod.OPENROUTER_API_KEY or "")}')


In [ ]:
# ── 1c. Validate step quality (chain 1) ─────────────────────────────────────────
# Spot-checks the first LLM chain to confirm the API call succeeded.
checks = {
    'At least 3 steps returned':         len(STEPS) >= 3,
    'All steps non-empty strings':        all(isinstance(s, str) and len(s.strip()) > 0 for s in STEPS),
    'All steps > 15 chars (not trivial)': all(len(s) > 15 for s in STEPS),
    'Real LLM was called (not fallback)': PROVIDER != 'local',
}

all_pass = True
for name, result in checks.items():
    icon = '✓' if result else '✗'
    print(f'  {icon}  {name}')
    if not result:
        all_pass = False

print()
print('Stage 1:', 'PASS ✓' if all_pass else 'WARN — check API key')


In [ ]:
# ── 1d. Define prewritten test cases (batch mode) ───────────────────────────
#
# 5 hand-crafted CoTs covering specific leakage patterns.
# Each has an "expect" label describing what the pipeline SHOULD detect.
#
# CONTROL            → all signals LOW  (no leakage, negative control)
# GRADUAL_LEAKAGE    → UMLS density + specificity drop; NLI stays high (core case)
# CONTRADICTION      → NLI catches abrupt wrong claim (direction flip)
# VAGUENESS_ESCALATION → UMLS signals drop sharply; NLI misses it
# COMPOUNDING_OVERGEN → all signals, gradual progressive drift

PREWRITTEN_COTS = [
    {
        'id':       'control_metformin',
        'label':    'CONTROL — Correct metformin mechanism',
        'source':   'prewritten',
        'question': 'What is the mechanism by which metformin lowers blood glucose?',
        'expect':   'low_risk',
        'steps': [
            'Metformin is a biguanide drug that primarily acts on the liver.',
            'It inhibits mitochondrial complex I, which raises the AMP:ATP ratio in hepatocytes.',
            'The elevated AMP:ATP ratio activates AMP-activated protein kinase (AMPK).',
            'AMPK activation suppresses SREBP-1c and downregulates gluconeogenic enzymes PEPCK and G6Pase.',
            'Reduced gluconeogenesis lowers hepatic glucose output, directly decreasing fasting blood glucose.',
            'Metformin also enhances GLUT4-mediated glucose uptake in skeletal muscle, improving peripheral insulin sensitivity.',
            'Together, these mechanisms lower HbA1c without stimulating insulin release or causing hypoglycemia.',
        ],
    },
    {
        'id':       'gradual_leakage_aspirin',
        'label':    'GRADUAL LEAKAGE — Aspirin overgeneralization',
        'source':   'prewritten',
        'question': 'How does aspirin prevent myocardial infarction?',
        'expect':   'umls_drop',
        'steps': [
            'Aspirin irreversibly acetylates and inhibits cyclooxygenase-1 (COX-1) in platelets.',
            'COX-1 inhibition blocks the conversion of arachidonic acid to thromboxane A2.',
            'Reduced thromboxane A2 impairs platelet activation and aggregation at sites of vascular injury.',
            'Decreased platelet aggregation reduces the likelihood of occlusive arterial thrombus formation.',
            'This antiplatelet effect broadly reduces the tendency for blood to clot in vessels.',
            'Therefore aspirin provides general protection against clotting-related cardiovascular conditions.',
            'Aspirin can consequently be considered a broad protective agent against most vascular diseases.',
        ],
    },
    {
        'id':       'contradiction_statins',
        'label':    'CONTRADICTION — Statin mechanism with abrupt wrong claim',
        'source':   'prewritten',
        'question': 'Do statins reduce LDL cholesterol?',
        'expect':   'nli_contradiction',
        'steps': [
            'Statins competitively inhibit HMG-CoA reductase, the rate-limiting enzyme in the mevalonate pathway.',
            'Inhibiting HMG-CoA reductase reduces endogenous cholesterol synthesis in hepatocytes.',
            'Lower intracellular cholesterol triggers upregulation of LDL receptors on hepatocyte surfaces.',
            'Increased LDL receptor density enhances clearance of LDL particles from the bloodstream.',
            'However, statins do not significantly lower LDL cholesterol levels in clinical practice.',
            'The modest reduction in hepatic synthesis is quickly compensated by increased intestinal cholesterol absorption.',
        ],
    },
    {
        'id':       'vagueness_escalation_insulin',
        'label':    'VAGUENESS ESCALATION — Insulin resistance molecular → vague',
        'source':   'prewritten',
        'question': 'How does insulin resistance develop in type 2 diabetes?',
        'expect':   'umls_drop_sharp',
        'steps': [
            'Insulin binds to its receptor on skeletal muscle, triggering autophosphorylation of the receptor tyrosine kinase domain.',
            'The activated receptor phosphorylates IRS-1 at specific tyrosine residues, creating docking sites for downstream proteins.',
            'Phosphorylated IRS-1 recruits PI3K, which phosphorylates PIP2 to generate PIP3 at the plasma membrane.',
            'PIP3 recruits PDK1 and Akt, leading to Akt phosphorylation and downstream activation of GLUT4 vesicle translocation.',
            'In type 2 diabetes, this signaling cascade becomes progressively impaired at multiple steps.',
            'The impairment leads to a reduced cellular response to normal circulating insulin levels.',
            'This reduced response results in elevated blood glucose and broader metabolic dysfunction.',
        ],
    },
    {
        'id':       'compounding_overgen_af',
        'label':    'COMPOUNDING OVERGENERALIZATION — AF anticoagulation cascade',
        'source':   'prewritten',
        'question': 'Should patients with atrial fibrillation take anticoagulants?',
        'expect':   'all_signals',
        'steps': [
            'Atrial fibrillation causes disorganized atrial electrical activity and loss of effective atrial contraction.',
            'Blood stasis in the left atrial appendage during AF promotes local thrombus formation.',
            'Thrombi originating in the left atrial appendage can embolize to cerebral arteries, causing ischemic stroke.',
            'Direct oral anticoagulants such as apixaban and rivaroxaban reduce stroke risk in AF patients by approximately 65%.',
            'Since anticoagulation reduces thromboembolic events, all patients with irregular heart rhythms should receive it.',
            'Any patient diagnosed with a cardiac arrhythmia is at elevated thromboembolic risk and warrants anticoagulation therapy.',
            'Anticoagulation is therefore broadly indicated for patients with cardiac disease to prevent adverse vascular events.',
        ],
    },
]

if BATCH_TEST:
    ALL_CHAINS = LLM_CHAINS + PREWRITTEN_COTS
    print(f'Batch mode: {len(LLM_CHAINS)} LLM chains + {len(PREWRITTEN_COTS)} prewritten = {len(ALL_CHAINS)} total\n')
    for c in ALL_CHAINS:
        src = '🤖 LLM' if c['source'] == 'llm' else '📝 Prewritten'
        print(f'  {src}  [{c["id"]}]  expect={c["expect"]}')
        print(f'         {c["label"]}')
else:
    ALL_CHAINS = []
    print('Single mode — prewritten cases available but not processed.')
    print('Enable batch mode in Stage 1a to run all 10 chains.')

# Always initialise so Stage 6e never raises NameError
BATCH_RESULTS = []


## Stage 2 — Concept Extractor

Extracts biomedical surface candidates (n-grams, acronyms) from each step
and links them to UMLS concepts (CUIs) if the UMLS API is configured.

| Cell | What it does |
|------|--------------|
| **2a** | Extract concepts for all 10 chains |
| **2b** | Per-chain concept summary table |


In [ ]:
# ── 2a. Concept extraction — all chains ─────────────────────────────────
import re as _re

def _strip_md(text):
    text = _re.sub(r'^#+\s*', '', text, flags=_re.MULTILINE)
    text = _re.sub(r'\*\*(.+?)\*\*', r'\1', text)
    text = _re.sub(r'\*(.+?)\*', r'\1', text)
    text = _re.sub(r'`(.+?)`', r'\1', text)
    return text.strip()

print(f'Extracting concepts for all {len(ALL_CHAINS)} chains...')
print('=' * 70)

for ci, chain in enumerate(ALL_CHAINS):
    steps_c = [_strip_md(s) for s in chain['steps']]
    chain['steps_clean'] = steps_c

    t0 = time.time()
    try:
        concepts = extract_concepts(steps_c, scispacy_when='never', top_k=5)
        chain['concepts'] = concepts
        valid   = sum(1 for step in concepts for c in step if c.get('valid'))
        elapsed = round(time.time() - t0, 2)
        lbl     = chain['label'][:55]
        n_steps = len(steps_c)
        print(f'  [{ci+1}/{len(ALL_CHAINS)}] {lbl}')
        print(f'         {valid} valid concepts across {n_steps} steps  ({elapsed}s)')
    except Exception as e:
        chain['concepts'] = [[] for _ in steps_c]
        print(f'  [{ci+1}/{len(ALL_CHAINS)}] ERROR: {e}')

print()
print('=' * 70)
n_ready = sum(1 for c in ALL_CHAINS if 'concepts' in c)
print(f'Concepts ready for {n_ready}/{len(ALL_CHAINS)} chains.')

# Set chain-1 variables for Stage 6 drill-down visualisations
STEPS_CLEAN = ALL_CHAINS[0]['steps_clean']
CONCEPTS    = ALL_CHAINS[0]['concepts']
print()
print('STEPS_CLEAN and CONCEPTS set to chain 1 for Stage 6 visualisations.')


In [ ]:
# ── 2b. Per-chain concept summary ──────────────────────────────────────────
rows = []
for chain in ALL_CHAINS:
    concepts = chain.get('concepts', [])
    total    = sum(len(step) for step in concepts)
    valid    = sum(1 for step in concepts for c in step if c.get('valid'))
    lbl      = chain['label']
    rows.append({
        'Chain':  lbl[:50] + '...' if len(lbl) > 50 else lbl,
        'Source': chain['source'],
        'Steps':  len(chain.get('steps_clean', chain['steps'])),
        'Total':  total,
        'Valid':  valid,
    })

df_concepts = pd.DataFrame(rows)
display(df_concepts.to_string(index=False))

if not os.environ.get('UMLS_API_KEY', ''):
    print()
    print('  UMLS not configured — all concept counts are 0. Set UMLS_API_KEY in Stage 0c.')


## Stage 3 — NLI Entailment + Guard Signals

Scores every adjacent step-pair for entailment / neutral / contradiction
across all chains, then derives qualitative guard tags on each pair.

| Cell | What it does |
|------|--------------|
| **3a** | Run NLI + guards for all 10 chains |
| **3b** | NLI probability table — all chains, all pairs |
| **3c** | Guard signal table — all chains, all pairs |

**Guard tags** catch cases where the NLI score alone may be misleading:

| Guard | Fires when |
|-------|------------|
| `lexical_duplicate` | Steps are ≥ 90% the same words — chain is repeating itself |
| `caution_band` | Top two probabilities nearly tied — model is uncertain |
| `direction_conflict` | A→B says entailment but B→A says contradiction |


In [ ]:
# ── 3a. NLI + guard derivation — all chains ───────────────────────────────
print(f'Running NLI + guards for all {len(ALL_CHAINS)} chains...')
print('=' * 70)

for ci, chain in enumerate(ALL_CHAINS):
    lbl      = chain['label'][:55]
    steps_c  = chain.get('steps_clean', [_strip_md(s) for s in chain['steps']])
    concepts = chain.get('concepts', [[] for _ in steps_c])

    t0 = time.time()
    try:
        pairs = build_entailment_records(steps_c, concepts)
        chain['pairs'] = pairs

        guarded = []
        for p in pairs:
            i, j = p['step_pair']
            try:
                rev = build_entailment_records(
                    [steps_c[j], steps_c[i]],
                    [concepts[j] if j < len(concepts) else [],
                     concepts[i] if i < len(concepts) else []],
                )
                rev_probs = rev[0]['probs'] if rev else None
            except Exception:
                rev_probs = None
            guards = derive_guards(
                premise       = steps_c[i] if i < len(steps_c) else '',
                hypothesis    = steps_c[j] if j < len(steps_c) else '',
                probs         = p['probs'],
                reverse_probs = rev_probs,
                config        = GUARD_CFG,
            )
            guarded.append({**p, 'guards': guards, 'reverse_probs': rev_probs})
        chain['guarded_pairs'] = guarded

        elapsed      = round(time.time() - t0, 2)
        label_counts = Counter(p['final_label'] for p in pairs)
        guard_counts = Counter(g for p in guarded for g in p['guards'])
        print()
        print(f'  [{ci+1}/{len(ALL_CHAINS)}] {lbl}')
        print(f'         pairs={len(pairs)}  labels={dict(label_counts)}  guards={dict(guard_counts) or "none"}  ({elapsed}s)')
    except Exception as e:
        chain['pairs']         = []
        chain['guarded_pairs'] = []
        print(f'  [{ci+1}/{len(ALL_CHAINS)}] ERROR: {e}')

print()
print('=' * 70)
n_ready = sum(1 for c in ALL_CHAINS if 'pairs' in c)
print(f'NLI + guards done for {n_ready}/{len(ALL_CHAINS)} chains.')

# Set chain-1 variables for Stage 6 drill-down
PAIRS         = ALL_CHAINS[0]['pairs']
GUARDED_PAIRS = ALL_CHAINS[0]['guarded_pairs']
print()
print('PAIRS and GUARDED_PAIRS set to chain 1 for Stage 6 visualisations.')


In [ ]:
# ── 3b. NLI probability table — all chains, all pairs ──────────────────────
rows = []
for chain in ALL_CHAINS:
    cid   = chain['id']
    steps = chain.get('steps_clean', chain['steps'])
    for p in chain.get('pairs', []):
        i, j  = p['step_pair']
        probs = p.get('probs', {})
        prem  = steps[i][:45] + '...' if len(steps[i]) > 45 else steps[i]
        hyp   = steps[j][:45] + '...' if len(steps[j]) > 45 else steps[j]
        rows.append({
            'Chain':      cid,
            'Pair':       f'{i}→{j}',
            'Premise':    prem,
            'Hypothesis': hyp,
            'P(entail)':  round(probs.get('entailment',    0), 3),
            'P(neutral)': round(probs.get('neutral',       0), 3),
            'P(contra)':  round(probs.get('contradiction', 0), 3),
            'Label':      p.get('final_label', '?'),
        })

df_nli = pd.DataFrame(rows)

def _colour_label(val):
    colours = {'contradiction': '#ffcccc', 'entailment': '#ccffcc', 'neutral': '#e8e8e8'}
    return 'background-color: ' + colours.get(val, 'white')

display(
    df_nli.style
          .applymap(_colour_label, subset=['Label'])
          .format({'P(entail)': '{:.3f}', 'P(neutral)': '{:.3f}', 'P(contra)': '{:.3f}'})
)


In [ ]:
# ── 3c. Guard signal table — all chains, all pairs ──────────────────────────
rows = []
for chain in ALL_CHAINS:
    cid = chain['id']
    for p in chain.get('guarded_pairs', []):
        i, j   = p['step_pair']
        probs  = p['probs']
        rprobs = p.get('reverse_probs') or {}
        rev_e  = round(rprobs.get('entailment', 0), 3) if rprobs else '—'
        rows.append({
            'Chain':         cid,
            'Pair':          f'{i}→{j}',
            'Label':         p['final_label'],
            'P(contra) fwd': round(probs.get('contradiction', 0), 3),
            'P(entail) rev': rev_e,
            'Guards':        ', '.join(p['guards']) or 'none',
        })

if rows:
    df_guards = pd.DataFrame(rows)
    display(df_guards.to_string(index=False))
    all_fired = Counter(
        g for chain in ALL_CHAINS
        for p in chain.get('guarded_pairs', [])
        for g in p['guards']
    )
    print()
    print(f'All guards across all chains: {dict(all_fired) or "none"}')
else:
    print('No guard data — run Stage 3a first.')


## Stage 5 — UMLS Signal Computation

Runs the UMLS chain-level scorers on the concepts extracted in Stage 2.
No additional UMLS API calls are made here.

**Signals:**

| Signal | What it measures | Output |
|--------|-----------------|--------|
| **Density** | Valid UMLS concept count per word, per step | risk (0–1), slope, leakage onset step |
| **Specificity** | Average hierarchy depth of concepts per step | risk (0–1), depth slope, abstraction leaps |

Both signals require `UMLS_API_KEY`. Without it they return `configured: False` and scores are 0.

**Cells:**

| Cell | What it does |
|------|--------------|
| **5a** | Density + specificity for all 10 chains; assembles `BATCH_RESULTS` for Stage 6e |
| **5b** | Per-chain UMLS signal summary table |


In [ ]:
# ── 5a. UMLS signal computation + BATCH_RESULTS assembly — all chains ──────────
print(f'Running UMLS scorers for all {len(ALL_CHAINS)} chains...')
print('=' * 70)

_empty_den  = {'configured': False, 'overall_risk': 0.0, 'per_step': [],
               'slope': None, 'leakage_onset_step': None}
_empty_spec = {'configured': False, 'overall_specificity_score': 0.0, 'per_step': [],
               'depth_slope': None, 'abstraction_leaps': []}

for ci, chain in enumerate(ALL_CHAINS):
    lbl      = chain['label'][:55]
    steps_c  = chain.get('steps_clean', [_strip_md(s) for s in chain['steps']])
    concepts = chain.get('concepts', [[] for _ in steps_c])

    t0 = time.time()
    try:
        density     = score_density(concepts, steps=steps_c)
        specificity = score_specificity(concepts)
        chain['density']     = density
        chain['specificity'] = specificity
        elapsed  = round(time.time() - t0, 2)
        den_risk = density.get('overall_risk', 0.0)
        spc_risk = specificity.get('overall_specificity_score', 0.0)
        onset    = density.get('leakage_onset_step')
        leaps    = len(specificity.get('abstraction_leaps', []))
        print()
        print(f'  [{ci+1}/{len(ALL_CHAINS)}] {lbl}')
        print(f'         density={den_risk:.3f}  specificity={spc_risk:.3f}  onset={onset}  leaps={leaps}  ({elapsed}s)')
    except Exception as e:
        chain['density']     = dict(_empty_den)
        chain['specificity'] = dict(_empty_spec)
        print(f'  [{ci+1}/{len(ALL_CHAINS)}] ERROR: {e}')

print()
print('=' * 70)
n_ready = sum(1 for c in ALL_CHAINS if 'density' in c)
print(f'UMLS scorers done for {n_ready}/{len(ALL_CHAINS)} chains.')

# ── Assemble BATCH_RESULTS ──────────────────────────────────────────────────────────────────
BATCH_RESULTS = []
for chain in ALL_CHAINS:
    if 'pairs' not in chain or 'density' not in chain:
        continue
    pairs   = chain['pairs']
    den     = chain['density']
    spec    = chain['specificity']
    avg_ent = (
        sum(p['probs'].get('entailment', 0.0) for p in pairs) / len(pairs)
        if pairs else 0.0
    )
    BATCH_RESULTS.append({
        'ok':                True,
        'id':                chain['id'],
        'label':             chain['label'],
        'source':            chain['source'],
        'expect':            chain['expect'],
        'concepts':          chain.get('concepts', []),
        'pairs':             pairs,
        'density':           den,
        'specificity':       spec,
        'nli_chain_risk':    round(1.0 - avg_ent, 3),
        'has_contradiction': any(p['final_label'] == 'contradiction' for p in pairs),
        'density_risk':      den.get('overall_risk', 0.0),
        'specificity_risk':  spec.get('overall_specificity_score', 0.0),
        'leakage_onset_step': den.get('leakage_onset_step'),
    })

print()
print(f'BATCH_RESULTS assembled: {len(BATCH_RESULTS)} chains ready for Stage 6.')

# Set chain-1 variables for Stage 6 drill-down
DENSITY     = ALL_CHAINS[0]['density']
SPECIFICITY = ALL_CHAINS[0]['specificity']
print('DENSITY and SPECIFICITY set to chain 1 for Stage 6 visualisations.')


In [ ]:
# ── 5b. Per-chain UMLS signal summary ────────────────────────────────────────
rows = []
for chain in ALL_CHAINS:
    den  = chain.get('density',     {})
    spec = chain.get('specificity', {})
    lbl  = chain['label']
    n_leaps = len(spec.get('abstraction_leaps', []))
    den_sl  = den.get('slope')
    spc_sl  = spec.get('depth_slope')
    rows.append({
        'Chain':      lbl[:45] + '...' if len(lbl) > 45 else lbl,
        'Source':     chain['source'],
        'Expect':     chain['expect'],
        'Den risk':   round(den.get('overall_risk', 0.0), 3),
        'Den slope':  round(den_sl, 4) if den_sl is not None else '—',
        'Onset':      den.get('leakage_onset_step', '—'),
        'Spec risk':  round(spec.get('overall_specificity_score', 0.0), 3),
        'Dep slope':  round(spc_sl, 4) if spc_sl is not None else '—',
        'Leaps':      n_leaps,
    })

df_umls = pd.DataFrame(rows)
display(df_umls.to_string(index=False))

if rows and not ALL_CHAINS[0].get('density', {}).get('configured', False):
    print()
    print('  UMLS not configured — scores are 0. Set UMLS_API_KEY in Stage 0c.')


## Stage 6 — Signal Alignment & Visualisation

Combines all three signals into four plots.
6a–6d use chain 1 as a concrete drill-down example;
6e shows the full comparison across all 10 chains.

| Cell | What it shows |
|------|--------------|
| **6a Timeline** | All three signals per step — where they rise and fall together |
| **6b Heatmap** | Step × signal risk grid — which steps are flagged by which signals |
| **6c Chain grades** | One bar per signal — overall verdict on chain 1 |
| **6d Contradiction matrix** | P(contradiction) for every step-pair in chain 1 |
| **6e Batch comparison** | Risk scores for all 10 chains side by side |


In [ ]:
# ── 6a. Per-step signal timeline ────────────────────────────────────────────
#
# NLI is pairwise (N-1 scores). We assign each pair score to its destination
# step j:  nli_risk[j] = 1 - P(entailment) for pair (j-1 → j).
# Step 0 has no incoming pair so it gets None.

n = len(STEPS)

nli_risk = [None] + [
    1.0 - p['probs'].get('entailment', 0.0)
    for p in PAIRS
]

density_vals = [
    next((r['density'] for r in DENSITY['per_step'] if r['step_index'] == i), None)
    for i in range(n)
]

depth_vals = [
    next((r['avg_depth'] for r in SPECIFICITY['per_step'] if r['step_index'] == i), None)
    for i in range(n)
]

def _norm(values):
    clean = [v for v in values if v is not None]
    if not clean or max(clean) == min(clean):
        return values
    lo, hi = min(clean), max(clean)
    return [None if v is None else (v - lo) / (hi - lo) for v in values]

nli_norm     = _norm(nli_risk)
density_norm = _norm(density_vals)
depth_norm   = _norm(depth_vals)

xs = list(range(n))

fig, ax = plt.subplots(figsize=(max(8, n * 1.2), 4))

nli_xs = [x for x, v in zip(xs, nli_norm) if v is not None]
nli_ys = [v for v in nli_norm if v is not None]
ax.plot(nli_xs, nli_ys, 'o-', color='#C44E52', linewidth=2, label='NLI risk (1 − P(entail))')

if DENSITY['configured']:
    d_xs = [x for x, v in zip(xs, density_norm) if v is not None]
    d_ys = [v for v in density_norm if v is not None]
    ax.plot(d_xs, d_ys, 's--', color='#4C72B0', linewidth=2, label='UMLS density (normalised)')

if SPECIFICITY['configured']:
    sp_xs = [x for x, v in zip(xs, depth_norm) if v is not None]
    sp_ys = [v for v in depth_norm if v is not None]
    ax.plot(sp_xs, sp_ys, '^:', color='#55A868', linewidth=2, label='UMLS depth (normalised)')

onset = DENSITY.get('leakage_onset_step')
if onset is not None:
    ax.axvline(onset, color='red', linestyle='--', alpha=0.6, linewidth=1.5,
               label=f'Leakage onset (step {onset})')

for leap in SPECIFICITY.get('abstraction_leaps', []):
    ax.axvspan(leap['from_step'] - 0.3, leap['to_step'] + 0.3,
               alpha=0.12, color='orange',
               label=f'Abstraction leap ({leap["from_step"]}→{leap["to_step"]})')

ax.set_xticks(xs)
ax.set_xticklabels([f'Step {i}' for i in xs], rotation=30, ha='right', fontsize=8)
ax.set_ylabel('Risk (0 = low, 1 = high)', fontsize=9)
ax.set_title('Per-step Signal Timeline\n(all signals normalised 0–1 for comparison)', fontsize=11)
ax.set_ylim(-0.05, 1.15)
ax.legend(fontsize=8, loc='upper left')
ax.grid(axis='y', alpha=0.3)

if not DENSITY['configured']:
    ax.text(0.99, 0.97, 'UMLS not configured — density + depth lines absent',
            transform=ax.transAxes, ha='right', va='top',
            fontsize=7, color='grey', style='italic')

plt.tight_layout()
plt.show()


In [ ]:
# ── 6b. Signal agreement heatmap ────────────────────────────────────────────
#
# Rows = steps,  Columns = [NLI risk, Density risk, Specificity risk]
# All values normalised 0–1.  Red = high risk, green = low risk.

nli_col     = [0.0 if v is None else v for v in nli_norm]
density_col = [0.0 if v is None else v for v in density_norm]
depth_col   = [0.0 if v is None else v for v in depth_norm]

density_risk_col = [1.0 - v for v in density_col] if DENSITY['configured'] \
                   else [0.0] * n
depth_risk_col   = [1.0 - v for v in depth_col]   if SPECIFICITY['configured'] \
                   else [0.0] * n

hm_data    = np.array([nli_col, density_risk_col, depth_risk_col]).T
col_labels = ['NLI risk', 'Density risk', 'Specificity risk']
row_labels = [f'Step {i}' for i in range(n)]

fig, ax = plt.subplots(figsize=(5, max(3, n * 0.55)))
sns.heatmap(
    hm_data,
    ax=ax,
    xticklabels=col_labels,
    yticklabels=row_labels,
    cmap='RdYlGn_r',
    vmin=0, vmax=1,
    annot=True, fmt='.2f',
    linewidths=0.5,
    cbar_kws={'label': 'Risk (0=low, 1=high)', 'shrink': 0.8},
)
ax.set_title('Signal Agreement Heatmap\n(each cell = risk score per step per signal)', fontsize=10)
ax.set_ylabel('Reasoning step', fontsize=9)

if not DENSITY['configured']:
    ax.text(0.99, -0.08, 'Density + Specificity columns are zero (UMLS not configured)',
            transform=ax.transAxes, ha='right', fontsize=7, color='grey', style='italic')

plt.tight_layout()
plt.show()


In [ ]:
# ── 6c. Chain grade summary bars ────────────────────────────────────────────

avg_entailment = (
    sum(p['probs'].get('entailment', 0.0) for p in PAIRS) / len(PAIRS)
    if PAIRS else 0.0
)
nli_chain_risk    = round(1.0 - avg_entailment, 3)
density_grade     = DENSITY.get('overall_risk', 0.0)
specificity_grade = SPECIFICITY.get('overall_specificity_score', 0.0)

signals = ['NLI chain risk', 'UMLS density risk', 'UMLS specificity risk']
scores  = [nli_chain_risk, density_grade, specificity_grade]

def _bar_color(v):
    if v < 0.3:   return '#2ca02c'
    if v < 0.6:   return '#ff7f0e'
    return '#d62728'

colors = [_bar_color(v) for v in scores]

fig, ax = plt.subplots(figsize=(7, 2.8))
bars = ax.barh(signals, scores, color=colors, height=0.45, edgecolor='white')

for bar, v in zip(bars, scores):
    ax.text(min(v + 0.02, 0.97), bar.get_y() + bar.get_height() / 2,
            f'{v:.3f}', va='center', fontsize=10, fontweight='bold')

ax.axvline(0.3, color='grey', linestyle=':', linewidth=1, alpha=0.7)
ax.axvline(0.6, color='grey', linestyle=':', linewidth=1, alpha=0.7)
ax.set_xlim(0, 1.1)
ax.set_xlabel('Risk score (0 = low, 1 = high)', fontsize=9)
ax.set_title('Chain Grade Summary\n(green < 0.3 | orange 0.3–0.6 | red > 0.6)', fontsize=10)
ax.grid(axis='x', alpha=0.3)

if not DENSITY['configured']:
    ax.text(0.99, 0.04, 'UMLS bars are 0 — UMLS not configured',
            transform=ax.transAxes, ha='right', fontsize=7, color='grey', style='italic')

plt.tight_layout()
plt.show()

print(f'  NLI chain risk            : {nli_chain_risk}')
print(f'  UMLS density risk         : {density_grade}')
print(f'  UMLS specificity risk     : {specificity_grade}')


In [ ]:
# ── 6d. NLI P(contradiction) matrix ─────────────────────────────────────────
#
# N×N matrix where cell [i,j] = P(contradiction) for step pair i→j.
# Only adjacent pairs are scored; non-adjacent cells are grey (NaN).

mat = np.full((n, n), np.nan)
for p in PAIRS:
    i, j = p['step_pair']
    mat[i, j] = p['probs'].get('contradiction', 0)

fig, ax = plt.subplots(figsize=(max(5, n * 0.9), max(4, n * 0.8)))
im = ax.imshow(mat, vmin=0, vmax=1, cmap='RdYlGn_r', aspect='auto')

ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels([f'Step {i}' for i in range(n)], rotation=45, ha='right', fontsize=8)
ax.set_yticklabels([f'Step {i}' for i in range(n)], fontsize=8)
ax.set_xlabel('Hypothesis (step j)', fontsize=9)
ax.set_ylabel('Premise (step i)', fontsize=9)

for p in PAIRS:
    i, j = p['step_pair']
    val  = mat[i, j]
    lbl  = p['final_label'][0].upper()
    ax.text(j, i, f'{lbl}\n{val:.2f}',
            ha='center', va='center', fontsize=8,
            color='white' if val > 0.6 else 'black')

plt.colorbar(im, ax=ax, label='P(contradiction)', shrink=0.8)
ax.set_title('NLI P(contradiction) per Step-Pair\n(E=entailment, N=neutral, C=contradiction)', fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
# ── 6e. Batch summary — all 10 chains compared ──────────────────────────────
# Only runs in batch mode. Shows a signal comparison table and a grouped
# bar chart so you can see which chains triggered which signals.

if not BATCH_TEST or not BATCH_RESULTS:
    print('Batch mode not enabled or no results — skipping.')
    print('Enable batch mode in Stage 1a and re-run.')
else:
    ok_results = [r for r in BATCH_RESULTS if r.get('ok')]

    # ── Summary table ──────────────────────────────────────────────────────
    rows = []
    for r in ok_results:
        def _flag(val, configured=True):
            if not configured: return '—'
            return '🔴' if val > 0.5 else ('🟡' if val > 0.3 else '🟢')

        rows.append({
            'Chain':        r['label'][:52] + '…' if len(r['label']) > 52 else r['label'],
            'Source':       'LLM' if r['source'] == 'llm' else 'Prewritten',
            'Expected':     r['expect'],
            'NLI risk':     round(r['nli_chain_risk'], 3),
            'NLI flag':     _flag(r['nli_chain_risk']),
            'Density':      round(r['density_risk'], 3),
            'Den flag':     _flag(r['density_risk'], r['density']['configured']),
            'Specificity':  round(r['specificity_risk'], 3),
            'Spec flag':    _flag(r['specificity_risk'], r['specificity']['configured']),
            'Contradiction': r['has_contradiction'],
            'Onset step':   r['leakage_onset_step'] if r['leakage_onset_step'] is not None else '—',
        })

    df_batch = pd.DataFrame(rows)
    print('── Batch Results Summary ──\n')
    display(df_batch.to_string(index=False))
    print()
    print('  🟢 risk < 0.3 (clean)   🟡 risk 0.3–0.5 (caution)   🔴 risk > 0.5 (flagged)   — not configured')
    if ok_results and not ok_results[0]['density']['configured']:
        print()
        print('  ℹ  Density and Specificity are 0 / “—” — UMLS not configured.')
        print('     NLI signal is fully active. Add UMLS_API_KEY for UMLS columns.')

    # ── Bar chart: three signals per chain ────────────────────────────────
    labels    = [r['label'].split('—')[-1].strip()[:30] for r in ok_results]
    nli_vals  = [r['nli_chain_risk']  for r in ok_results]
    den_vals  = [r['density_risk']    for r in ok_results]
    spec_vals = [r['specificity_risk'] for r in ok_results]

    x     = np.arange(len(ok_results))
    width = 0.25
    fig, ax = plt.subplots(figsize=(max(12, len(ok_results) * 1.6), 5))

    ax.bar(x - width, nli_vals,  width, label='NLI risk',          color='#C44E52', alpha=0.85)
    ax.bar(x,          den_vals,  width, label='UMLS density risk', color='#4C72B0', alpha=0.85)
    ax.bar(x + width, spec_vals, width, label='UMLS specif. risk', color='#55A868', alpha=0.85)

    ax.axhline(0.3, color='grey', linestyle=':', linewidth=1, alpha=0.6)
    ax.axhline(0.5, color='red',  linestyle=':', linewidth=1, alpha=0.4)

    n_llm = sum(1 for r in ok_results if r['source'] == 'llm')
    if 0 < n_llm < len(ok_results):
        ax.axvline(n_llm - 0.5, color='black', linestyle='--', linewidth=1.2, alpha=0.5)
        ax.text(n_llm * 0.5 - 0.5,  ax.get_ylim()[1] * 0.95,
                'LLM',       ha='center', fontsize=8, color='grey')
        ax.text(n_llm + (len(ok_results) - n_llm) * 0.5 - 0.5,
                ax.get_ylim()[1] * 0.95,
                'Prewritten', ha='center', fontsize=8, color='grey')

    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('Risk score (0–1)', fontsize=9)
    ax.set_ylim(0, 1.1)
    ax.set_title('Batch Signal Comparison — All 10 Chains\n'
                 '(dashed lines: 0.3 caution threshold, 0.5 flag threshold)', fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.show()
